In [56]:
import pandas as pd

In [57]:
cafeSales=pd.read_csv("dirty_cafe_sales.csv")

In [58]:
cafeSales.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [59]:
cafeSales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB


In [60]:
df_e=cafeSales[(cafeSales=="ERROR")|(cafeSales=="UNKNOWN")].dropna(how='all')

In [61]:
df_e


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
2,NaN,NaN,NaN,NaN,ERROR,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,UNKNOWN,UNKNOWN,NaN
6,NaN,UNKNOWN,NaN,NaN,NaN,ERROR,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,UNKNOWN,NaN
11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ERROR
...,...,...,...,...,...,...,...,...
9985,NaN,NaN,NaN,NaN,NaN,NaN,UNKNOWN,NaN
9988,NaN,NaN,NaN,NaN,NaN,ERROR,NaN,NaN
9992,NaN,NaN,NaN,NaN,NaN,UNKNOWN,NaN,NaN
9994,NaN,UNKNOWN,NaN,NaN,NaN,NaN,NaN,NaN


In [62]:
import numpy as np




In [63]:
cafeSales['Total Spent']=cafeSales['Total Spent'].replace(['ERROR','UNKNOWN'],np.nan)	

In [64]:
cafeSales['Total Spent']=pd.to_numeric(cafeSales['Total Spent'],errors='coerce')

In [65]:
cafeSales['Quantity'] = pd.to_numeric(cafeSales['Quantity'], errors='coerce')
cafeSales['Price Per Unit'] = pd.to_numeric(cafeSales['Price Per Unit'], errors='coerce')

In [66]:
cafeSales['Total Spent'] = cafeSales['Total Spent'].fillna(cafeSales['Quantity'] * cafeSales['Price Per Unit'])

In [67]:
cafeSales['Payment Method'] = cafeSales['Payment Method'].replace('UNKNOWN', np.nan)
cafeSales['Location'] = cafeSales['Location'].replace('UNKNOWN', np.nan)

In [68]:

cafeSales.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,NaN,NaN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [69]:
print(cafeSales[['Payment Method','Location']].isna().sum())

Payment Method    2872
Location          3603
dtype: int64


In [70]:
from sklearn.preprocessing import LabelEncoder

In [71]:
ml_data = cafeSales.copy()

In [72]:
le_item = LabelEncoder()
ml_data['Item_Encoded'] = le_item.fit_transform(ml_data['Item'].astype(str))

In [73]:
ml_data['Transaction Date'] = pd.to_datetime(ml_data['Transaction Date'], errors='coerce')
ml_data['Year'] = ml_data['Transaction Date'].dt.year
ml_data['Month'] = ml_data['Transaction Date'].dt.month
ml_data['Day'] = ml_data['Transaction Date'].dt.day

In [74]:
ml_data['Year'] = ml_data['Year'].fillna(ml_data['Year'].mode()[0])
ml_data['Month'] = ml_data['Month'].fillna(ml_data['Month'].mode()[0])
ml_data['Day'] = ml_data['Day'].fillna(ml_data['Day'].mode()[0])
ml_data[['Item', 'Item_Encoded', 'Year', 'Month', 'Day']].head()

,Item,Item_Encoded,Year,Month,Day
0,Coffee,1,2023.0,9.0,8.0
1,Cake,0,2023.0,5.0,16.0
2,Cookie,2,2023.0,7.0,19.0
3,Salad,5,2023.0,4.0,27.0
4,Coffee,1,2023.0,6.0,11.0


In [75]:
features = ['Item_Encoded', 'Quantity', 'Price Per Unit', 'Total Spent', 'Year', 'Month', 'Day']

In [76]:
train_set = ml_data[ml_data['Payment Method'] != 'Unknown']
predict_set = ml_data[ml_data['Payment Method'] == 'Unknown']

In [77]:
le_payment = LabelEncoder()
X_train = train_set[features]
y_train = le_payment.fit_transform(train_set['Payment Method'])

In [78]:
print(f"عدد الصفوف الجاهزة لتدريب النموذج: {X_train.shape[0]}")
print(f"عدد الصفوف المفقودة التي سنتنبأ بها: {predict_set.shape[0]}")

عدد الصفوف الجاهزة لتدريب النموذج: 10000
عدد الصفوف المفقودة التي سنتنبأ بها: 0


In [79]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [80]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_split, y_train_split)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [81]:
y_pred = model.predict(X_val_split)
accuracy = accuracy_score(y_val_split, y_pred)

print(f" دقة النموذج في توقع طريقة الدفع الصحيحة هي: {accuracy * 100:.2f}%")

 دقة النموذج في توقع طريقة الدفع الصحيحة هي: 25.40%


In [82]:
cafeSales['Expected_Total'] = cafeSales['Quantity'] * cafeSales['Price Per Unit']
anomalies = cafeSales[abs(cafeSales['Total Spent'] - cafeSales['Expected_Total']) > 0.01]
print(f" تم اكتشاف {len(anomalies)} معاملة تحتوي على أخطاء حسابية أو شاذة في النظام")
anomalies.head()

 تم اكتشاف 0 معاملة تحتوي على أخطاء حسابية أو شاذة في النظام


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date,Expected_Total


In [83]:
from sklearn.ensemble import IsolationForest
anomaly_features = ['Quantity', 'Price Per Unit', 'Total Spent']
X_anomaly = cafeSales[anomaly_features].fillna(0) 
iso_forest = IsolationForest(contamination=0.01, random_state=42)
cafeSales['Anomaly_Score'] = iso_forest.fit_predict(X_anomaly)
ml_anomalies = cafeSales[cafeSales['Anomaly_Score'] == -1]

print(f" باستخدام تعلم الآلة (Isolation Forest)، تم اكتشاف {len(ml_anomalies)} صف يحتوي على سلوك شراء شاذ في الكافيه")
ml_anomalies[anomaly_features].head()

 باستخدام تعلم الآلة (Isolation Forest)، تم اكتشاف 72 صف يحتوي على سلوك شراء شاذ في الكافيه


,Quantity,Price Per Unit,Total Spent
20,NaN,4.0,20.0
177,NaN,5.0,25.0
214,NaN,5.0,25.0
629,NaN,NaN,12.0
777,NaN,5.0,25.0


Conclusion & Key Founds
* Data Cleaning:
* 




